<a href="https://colab.research.google.com/github/Sameenakazi/github-resume-generator/blob/main/project_2_Context_aware_customer_support_prompt_system_flow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#@title 1. Install Dependencies
!pip install -q groq gradio

Developer view: Installs the two libraries needed — groq for API calls, gradio for the UI. -q keeps output quiet.

User view: One-time setup so Colab has everything it needs to run the project.

In [2]:
#@title 2. Setup Groq API Client
import os
from getpass import getpass
from groq import Groq

GROQ_API_KEY = getpass("Enter your Groq API Key: ")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

client = Groq(api_key=GROQ_API_KEY)
MODEL = "openai/gpt-oss-120b"

print("Groq client initialized with model:", MODEL)

Enter your Groq API Key: ··········
Groq client initialized with model: openai/gpt-oss-120b


Developer view: getpass hides the key while typing so it's never printed or saved in the notebook. Client is initialized once and reused by every agent.
User view: You'll be prompted to paste your free Groq API key — it's never shown on screen or saved to the file.

In [3]:
#@title 3. Core LLM Call Helper
import json

def call_llm(system_prompt, user_prompt, json_mode=True, temperature=0.3):
    """Sends a system+user prompt to Groq and returns parsed output."""
    kwargs = dict(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
    )
    if json_mode:
        kwargs["response_format"] = {"type": "json_object"}

    response = client.chat.completions.create(**kwargs)
    content = response.choices[0].message.content

    if json_mode:
        try:
            return json.loads(content)
        except json.JSONDecodeError:
            return {"error": "Failed to parse JSON", "raw": content}
    return content

Developer view: One reusable function every agent calls. response_format={"type": "json_object"} forces Groq to return valid JSON, so downstream code never has to regex-parse text. A try/except guards against rare malformed JSON.
User view: This is the "engine" all five agents plug into — you won't call it directly, but everything runs through it.

In [4]:
#@title 4. Agent 1 — Sentiment & Intent Classifier
CLASSIFIER_SYSTEM_PROMPT = """
You are a Customer Support Query Classifier Agent.

Your ONLY job is to analyze the customer's message and classify it.

Classify the query into exactly one category:
- "billing"    -> payments, invoices, charges, refunds, subscriptions
- "technical"  -> bugs, errors, login issues, "how do I" questions, app/product malfunction
- "complaint"  -> dissatisfaction, repeated issues, poor service, frustration with the company
- "general"    -> anything that does not clearly fit the above

Also detect the customer's sentiment:
- "positive", "neutral", "frustrated", or "angry"

Respond ONLY with valid JSON in this exact schema, no extra text:
{
  "category": "billing | technical | complaint | general",
  "sentiment": "positive | neutral | frustrated | angry",
  "confidence": 0.0 to 1.0
}
"""

def classifier_agent(query):
    return call_llm(CLASSIFIER_SYSTEM_PROMPT, query, json_mode=True, temperature=0.1)

Developer view: Low temperature (0.1) keeps classification consistent/deterministic — you don't want creative guessing here, just reliable routing signal.
User view: This agent "reads the room" first — figuring out what kind of query it is and how upset the customer sounds, before anything else happens.

In [5]:
#@title 5. Agent 3a — Billing Agent
BILLING_SYSTEM_PROMPT = """
You are the Billing Support Agent for a customer support system.

You ONLY handle billing, payment, invoice, refund, and subscription queries.
Your response MUST be structured, factual, and free of unnecessary filler.

Respond ONLY with valid JSON in this exact schema:
{
  "issue_summary": "one line summary of the billing issue",
  "account_action_needed": "what action needs to happen on the account, or 'none'",
  "next_steps": ["step 1", "step 2", "step 3"],
  "relevant_policy": "a short, plausible policy reference relevant to the issue"
}

Rules:
- Keep issue_summary under 20 words.
- next_steps must be concrete and actionable, 2 to 4 items.
- Do not apologize excessively, this is a structured/procedural response, not an empathetic one.
"""

def billing_agent(query):
    return call_llm(BILLING_SYSTEM_PROMPT, query, json_mode=True, temperature=0.3)

Developer view: Explicit schema + explicit anti-pattern ("do not apologize excessively") stops the model from drifting into the empathetic tone meant for the complaint agent — this is what satisfies "billing queries produce structured responses" from your problem statement.
User view: Handles anything about money — charges, refunds, invoices — with a clean, no-nonsense action list.

In [6]:
#@title 6. Agent 3b — Technical Support Agent
TECHNICAL_SYSTEM_PROMPT = """
You are the Technical Support Agent for a customer support system.

You ONLY handle bugs, errors, login issues, and "how do I" product questions.
Your response MUST be step-by-step troubleshooting instructions.

Respond ONLY with valid JSON in this exact schema:
{
  "prerequisites": ["thing to check/have before starting", "..."],
  "steps": [
    {"step_number": 1, "instruction": "...", "expected_outcome": "..."},
    {"step_number": 2, "instruction": "...", "expected_outcome": "..."}
  ],
  "fallback": "what to do if none of the steps resolve the issue"
}

Rules:
- Provide 3 to 6 concrete, numbered steps in logical order.
- Each step must have a clear expected_outcome so the user knows if it worked.
- fallback must offer a next channel (e.g., escalate to specialist, share logs, etc.)
"""

def technical_agent(query):
    return call_llm(TECHNICAL_SYSTEM_PROMPT, query, json_mode=True, temperature=0.3)

Developer view: Forcing expected_outcome per step is what makes this genuinely "step-by-step troubleshooting" rather than a vague paragraph — it gives the customer a way to self-verify each step.
User view: Walks the customer through fixing a bug or error one clear step at a time, telling them what should happen if it worked.

In [7]:
#@title 7. Agent 3c — Complaint / Empathy Agent
COMPLAINT_SYSTEM_PROMPT = """
You are the Complaint Resolution Agent for a customer support system.

You ONLY handle dissatisfaction, complaints, and emotionally charged queries.
Your response MUST lead with genuine empathy before offering any resolution.

Respond ONLY with valid JSON in this exact schema:
{
  "acknowledgment": "one sentence acknowledging what specifically went wrong",
  "empathy_statement": "one sentence genuinely validating the customer's frustration, no corporate deflection language like 'we value your feedback'",
  "resolution_steps": ["step 1", "step 2"],
  "tone_note": "brief note on how this should be delivered (e.g., calm, sincere, non-defensive)"
}

Rules:
- Never use generic phrases like "we apologize for any inconvenience".
- Be specific to what the customer described.
- resolution_steps must be realistic and customer-facing, 1 to 3 items.
"""

def complaint_agent(query):
    return call_llm(COMPLAINT_SYSTEM_PROMPT, query, json_mode=True, temperature=0.4)

Developer view: Slightly higher temperature (0.4) than billing/technical lets language feel more natural/human here, since empathy sounding templated defeats the purpose. Banning stock phrases forces specificity.
User view: This agent's whole job is to make an upset customer feel heard before jumping to a fix.

In [8]:
#@title 8. Router — Dispatch Logic
def router(category):
    """Pure logic, no LLM call. Maps a category to its specialist agent function."""
    routing_table = {
        "billing": billing_agent,
        "technical": technical_agent,
        "complaint": complaint_agent,
    }
    # default to complaint-style handling for 'general' so the user still gets a considerate reply
    return routing_table.get(category, complaint_agent)

Developer view: No LLM call, no cost — just a dictionary lookup. This is intentionally cheap and fast since it's a plumbing decision, not a reasoning task.
User view: This is the invisible "traffic controller" deciding which specialist handles your query.

In [9]:
#@title 9. Agent 4 — Escalation Detector Agent
ESCALATION_SYSTEM_PROMPT = """
You are the Escalation Detection Agent for a customer support system.

You independently review the customer's query and sentiment to decide if this
needs to be escalated to a human supervisor.

Escalate when you see: repeated/ongoing issues, threats of legal action or
cancellation, requests to speak to a manager, extreme anger, or mentions of
SLA/contract breaches.

Respond ONLY with valid JSON in this exact schema:
{
  "escalate": true or false,
  "reason": "short reason for the decision",
  "priority": "low | medium | high"
}
"""

def escalation_agent(query, sentiment):
    user_prompt = f"Customer query: {query}\nDetected sentiment: {sentiment}"
    return call_llm(ESCALATION_SYSTEM_PROMPT, user_prompt, json_mode=True, temperature=0.1)

Developer view: Runs independently of the specialist agent (takes raw query + sentiment, not the specialist's output) so it isn't biased by how the specialist framed the issue — a truer second opinion.
User view: A silent safety-net check running in the background to flag anything serious enough for a human to step in.

In [10]:
#@title 10. Agent 5 — Response Formatter Agent
FORMATTER_SYSTEM_PROMPT = """
You are the Final Response Formatter Agent for a customer support system.

You receive the specialist agent's structured JSON output, the customer's
sentiment, and an escalation flag. Your job is to turn this into a single,
polished, customer-facing reply in plain text (NOT JSON).

Formatting rules:
- If category is "billing": present it clearly with a short summary line
  followed by a numbered/bulleted action list. Professional, no-nonsense tone.
- If category is "technical": present it as clearly numbered troubleshooting
  steps, each with what to expect. Clear and instructional tone.
- If category is "complaint": lead with empathy and acknowledgment BEFORE any
  steps. Warm, sincere, non-defensive tone.
- Sentiment-aware tone: if sentiment is "frustrated" or "angry", soften all
  language, add a reassuring opening line, and avoid anything that sounds dismissive.
- If escalate is true, add a clearly marked escalation notice at the top,
  e.g. "⚠️ This has been flagged for priority review by our team."
- Keep the entire reply concise: under 150 words.
- Do not mention "JSON", "agent", or any internal system details to the customer.
"""

def formatter_agent(category, specialist_output, sentiment, escalation_result):
    user_prompt = f"""
Category: {category}
Sentiment: {sentiment}
Escalation: {json.dumps(escalation_result)}
Specialist Output: {json.dumps(specialist_output)}
"""
    return call_llm(FORMATTER_SYSTEM_PROMPT, user_prompt, json_mode=False, temperature=0.4)

Developer view: This is where "sentiment-aware response formatting" from your problem statement actually gets implemented — it's the only agent that sees sentiment + escalation + specialist output together, so it's the single place tone-blending happens.
User view: Takes everything the other agents figured out and turns it into the one clean message you'd actually receive.

In [11]:
#@title 11. Orchestrator — Full Pipeline
def run_pipeline(query):
    """Runs the full 5-stage pipeline and returns every intermediate result."""

    # Stage 1: Classify intent + sentiment
    classification = classifier_agent(query)
    category = classification.get("category", "general")
    sentiment = classification.get("sentiment", "neutral")

    # Stage 2: Route to correct specialist agent
    specialist_fn = router(category)

    # Stage 3: Run specialist agent
    specialist_output = specialist_fn(query)

    # Stage 4: Run escalation detector (independent of specialist output)
    escalation_result = escalation_agent(query, sentiment)

    # Stage 5: Format final response
    final_response = formatter_agent(category, specialist_output, sentiment, escalation_result)

    return {
        "classification": classification,
        "category": category,
        "specialist_output": specialist_output,
        "escalation_result": escalation_result,
        "final_response": final_response,
    }

Developer view: Ties all five agents together in the exact sequence from your flow diagram, and returns every intermediate result so the UI can display each agent's work separately — not just the final answer.
User view: This single function is "run the whole pipeline" — you give it one query, it runs everything behind the scenes and hands back the full trail plus the final reply.

In [12]:
# @title 12. Gradio UI — Hero Banner, Stepper, Color-Coded Agents, Light/Dark Toggle
import gradio as gr

gr.close_all()  # ensures a clean relaunch each time this cell runs

# ------------------------------------------------------------------
# 1) THEME — teal/amber palette (distinct from indigo/pink combos)
# ------------------------------------------------------------------
THEME = gr.themes.Soft(
    primary_hue="teal",
    secondary_hue="amber",
    neutral_hue="slate",
    radius_size="lg",
    font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif", "sans-serif"],
    font_mono=[gr.themes.GoogleFont("JetBrains Mono"), "ui-monospace", "monospace"],
).set(
    button_primary_background_fill="linear-gradient(90deg,#0d9488,#0891b2,#0369a1)",
    button_primary_background_fill_hover="linear-gradient(90deg,#0f766e,#0e7490,#075985)",
    button_primary_text_color="white",
    block_title_text_weight="700",
    block_shadow="0 4px 14px rgba(0,0,0,.08)",
)

# ------------------------------------------------------------------
# 2) CUSTOM CSS — hero banner, stepper, agent cards, final ticket card
# ------------------------------------------------------------------
CSS = """
.gradio-container {max-width: 1400px !important; margin: auto;
  background: linear-gradient(180deg,#f0fdfa 0%,#ffffff 45%);}
.dark .gradio-container {background: linear-gradient(180deg,#0a1f1c 0%,#0f2027 100%);}

/* Hero banner */
#hero {background: linear-gradient(135deg,#0d9488 0%,#0891b2 55%,#0369a1 100%);
  border-radius: 20px; padding: 26px 32px; box-shadow: 0 12px 30px rgba(13,148,136,.30);}
#hero h1 {margin:0; font-size:1.9rem; font-weight:800; color:#fff !important;}
#hero p  {margin:6px 0 0; color:#fff !important; opacity:.92;}

#hero .chips span {display:inline-block; background:rgba(255,255,255,.18); color:#fff;
  border-radius:999px; padding:4px 12px; margin:12px 8px 0 0; font-size:.78rem; font-weight:600;}

/* Pipeline stepper */
.stepper {display:flex; gap:8px; flex-wrap:wrap; margin:6px 0 10px;}
.step {flex:1; min-width:150px; text-align:center; padding:9px 10px; border-radius:12px;
  border:1px solid rgba(128,128,128,.3); font-size:.82rem; font-weight:600; opacity:.5;}
.step.done {opacity:1; border-color:#0d9488; background:rgba(13,148,136,.15); color:#0d9488;}

/* Color-coded agent panels, fixed height + internal scroll */
.agent-panel {height:300px !important; overflow-y:auto !important; border-radius:12px;
  padding:6px; border:1px solid rgba(128,128,128,.25);}
.agent-panel .json-holder, .agent-panel pre {max-height:260px; overflow-y:auto !important;}
.a1 {border-left:6px solid #0d9488 !important;}
.a2 {border-left:6px solid #0891b2 !important;}
.a3 {border-left:6px solid #f59e0b !important;}

/* Escalation banner colors */
.escalate-high {background:rgba(220,38,38,.12) !important; border-left:6px solid #dc2626 !important;}
.escalate-safe {background:rgba(13,148,136,.10) !important; border-left:6px solid #0d9488 !important;}

/* Final response — ticket-style card */
#final-card {background:#ffffff !important; border-radius:16px; padding:26px 30px !important;
  box-shadow:0 10px 28px rgba(0,0,0,.15); font-size:15px; line-height:1.6; border-top:5px solid #0369a1;}
#final-card * {color:#0f172a !important;}
"""

# ------------------------------------------------------------------
# 3) Stepper HTML — marks stages complete once pipeline finishes
# ------------------------------------------------------------------
STEP_NAMES = ["🧭 Classify", "🔀 Route", "🛠️ Specialist", "🚨 Escalation", "🎨 Format"]

def stepper_html(done=False):
    pills = "".join(
        f'<div class="step {"done" if done else ""}">{name}</div>' for name in STEP_NAMES
    )
    return f'<div class="stepper">{pills}</div>'

# ------------------------------------------------------------------
# 4) Wrapper around run_pipeline (from Cell 11) for UI formatting
# ------------------------------------------------------------------
def process_query(query):
    if not query or not query.strip():
        return stepper_html(False), {}, {}, {}, "", "Please enter a query."

    result = run_pipeline(query)
    escalate = result["escalation_result"].get("escalate", False)
    banner = "🚨 ESCALATED — priority review needed\n\n" if escalate else ""

    return (
        stepper_html(True),
        result["classification"],
        result["specialist_output"],
        result["escalation_result"],
        result["category"],
        banner + result["final_response"],
    )

# ------------------------------------------------------------------
# 5) LAYOUT
# ------------------------------------------------------------------
with gr.Blocks(theme=THEME, css=CSS, title="Context-Aware Customer Support System") as demo:

    gr.HTML("""
    <div id="hero">
      <h1>🎯 Context-Aware Customer Support Prompt System</h1>
      <p>5 specialized prompt agents classify, route, resolve, and format every customer query.</p>
      <div class="chips"><span>⚡ Groq</span><span>🧠 gpt-oss-120b</span><span>🤖 5 agents</span><span>📋 Structured JSON</span><span>💬 Sentiment-aware</span></div>
    </div>
    """)

    status_html = gr.HTML(stepper_html(False))

    with gr.Row():
        query_input = gr.Textbox(
            label="Customer Query", lines=2, scale=4,
            placeholder="e.g. I was charged twice this month and no one has responded to my emails...",
        )
        with gr.Column(scale=1):
            submit_btn = gr.Button("🚀 Run Pipeline", variant="primary", size="lg")
            mode_btn = gr.Button("🌗 Light / Dark", size="sm")

    gr.Markdown("### 🔍 Agent Pipeline — Live Outputs")
    with gr.Row():
        with gr.Column(elem_classes="agent-panel a1"):
            classifier_out = gr.JSON(label="① Classifier Agent")
        with gr.Column(elem_classes="agent-panel a2"):
            specialist_out = gr.JSON(label="② Specialist Agent")
        with gr.Column(elem_classes="agent-panel a3"):
            escalation_out = gr.JSON(label="③ Escalation Detector")
        with gr.Column(elem_classes="agent-panel"):
            category_out = gr.Textbox(label="Routed Category", lines=10)

    gr.Markdown("### ✅ Final Customer-Facing Response")
    final_out = gr.Textbox(label="", lines=6, elem_id="final-card")

    submit_btn.click(
        fn=process_query,
        inputs=query_input,
        outputs=[status_html, classifier_out, specialist_out, escalation_out, category_out, final_out],
    )
    mode_btn.click(fn=None, js="() => document.body.classList.toggle('dark')")

demo.launch(debug=False, share=True)

/tmp/ipykernel_943/709171409.py:99: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=THEME, css=CSS, title="Context-Aware Customer Support System") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c5c46a5d526f4d7fe3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Developer view: All four intermediate outputs sit in one gr.Row of equal-width columns, with the final response in its own row below — a fixed, flat layout with no nested scroll areas, so it fits on one screen. gr.JSON auto-pretty-prints each agent's structured output compactly.
User view: Type a query, hit Run Pipeline, and you see — side by side, no scrolling — exactly what each agent decided (classification, specialist reasoning, escalation check) plus the final polished reply at the bottom.